In [ ]:
import os
import json
import re
import pdfplumber
from PIL import Image
from pypdf import PdfReader, PdfWriter
from transformers import pipeline
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output, calculate_model_confidence
)

# =========================================================
# LOAD MODEL
# =========================================================
pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)

"""
DentaQuest-style EOB PDF processing:

1. crop_claim_tables()  -> crops every "Claim Detail" table (starts at the
   "Patient Name:" line, ends at the "Total:" row) into its own PNG.
   If a table runs off the bottom of the page and continues on the next
   page (same claim, patient header repeated), the two page-fragments are
   stitched into a single image automatically.

   When a table continues onto the next page, the top-page fragment is
   trimmed to the bottom of its LAST DATA ROW (not the page footer), and
   the continuation-page fragment is trimmed to start right after its
   repeated "Claim Detail" / patient header / column-header block (not
   from the very top of the page). This avoids sandwiching logo/footer/
   "Page X of Y"/repeated-header noise between two halves of the same
   table, which was confusing the VLM into dropping or merging rows.

2. rotate_pages()       -> rotates every page of the PDF EXCEPT the first
   two pages (the cover / payment-summary pages), leaving pages 1-2 as-is.
   Angle defaults to 90 (clockwise) -- change ROTATE_ANGLE below if your
   printer/viewer needs the opposite direction (use -90 or 270).
"""

# ----------------------------------------------------------------------
# Tunables
# ----------------------------------------------------------------------
DPI = 200                  # render resolution for the crops
BOTTOM_PADDING_PT = 6      # a little breathing room under the Total row
ROTATE_ANGLE = 90          # degrees, clockwise. Flip to -90/270 if needed
SKIP_PAGES = 2              # don't rotate the first N pages
LAST_ROW_PADDING_PT = 14    # breathing room under the last data row on a page that
                             # continues onto the next page (approx. one text line)


# ----------------------------------------------------------------------
# 1. Table cropping / cross-page merging
# ----------------------------------------------------------------------
def _last_row_bottom(page, region_top, region_bottom, line_pad=LAST_ROW_PADDING_PT):
    """On an INCOMPLETE segment (no 'Total:' found yet on this page), trim
    the crop to the bottom of the last actual data row instead of
    footer_top. footer_top grabs everything down to the 'Current Dental...'
    disclaimer line, which includes the DentaQuest logo, 'Page X of Y',
    and blank whitespace -- all of which sits between the two halves of
    the same table once stitched and confuses the VLM.

    Anchors on the same D#### service-code pattern already used by
    count_service_rows, then pads down by ~one line height so wrapped
    description text under the code isn't clipped.
    """
    words = page.extract_words()
    row_bottoms = [
        w["bottom"] for w in words
        if region_top <= w["top"] <= region_bottom and re.fullmatch(r"D\d{4}", w["text"])
    ]
    if row_bottoms:
        return max(row_bottoms) + line_pad
    return region_bottom  # fallback: old behavior if no D-code found


def _header_row_bottom(page, region_top, region_bottom, pad=2):
    """On a CONTINUATION page, find where the repeated 'Claim Detail'
    title + patient/provider block + column-header row ends, so the
    stitched crop can start right at the data rows instead of
    duplicating that whole header block a second time in the merged
    image.

    The column header wraps onto TWO physical lines:
        line 1: "... Plan   Processing"
        line 2: "... Pay    Policies"
    Anchoring on "Processing" alone only clears line 1, leaving line 2's
    fragments ("Code", "Service", "Insurance", "Policies", ...) poking
    into the crop just above the first continuation row. So anchor on
    BOTH "Processing" and "Policies" and take whichever sits lower --
    that clears the entire two-line header block.
    """
    words = page.extract_words()
    anchors = [
        w["bottom"] for w in words
        if region_top <= w["top"] <= region_bottom and w["text"] in ("Processing", "Policies")
    ]
    if anchors:
        return max(anchors) + pad
    return region_top  # fallback: no trimming if anchor not found


def _page_segments(page):
    """Find every claim table on a page: its patient/claim id, the pixel
    (point) top of the 'Patient Name:' line, and either the bottom of its
    'Total:' row (complete) or the trimmed last-row position (open / continues)."""
    words = page.extract_words(use_text_flow=False, keep_blank_chars=False)

    patient_markers = []
    for i, w in enumerate(words):
        if w["text"] == "Patient" and i + 1 < len(words) and words[i + 1]["text"] == "Name:":
            top = w["top"]
            line = [ww["text"] for ww in words if abs(ww["top"] - top) < 2]
            idx = line.index("Name:")
            name_parts = []
            for t in line[idx + 1:]:
                if t.endswith(":") or t == "Provider":
                    break
                name_parts.append(t)
            claim = next(
                (ww["text"] for ww in words
                 if abs(ww["top"] - top) < 3 and re.fullmatch(r"\d{12,18}", ww["text"])),
                None,
            )
            patient_markers.append({"top": top, "name": " ".join(name_parts), "claim": claim})

    total_markers = sorted(
        ({"top": w["top"], "bottom": w["bottom"]} for w in words if w["text"] == "Total:"),
        key=lambda d: d["top"],
    )

    footer_top = next(
        (words[i]["top"] for i in range(len(words) - 1)
         if words[i]["text"] == "Current" and words[i + 1]["text"].startswith("Dental")),
        page.height,
    )

    segments = []
    for i, pm in enumerate(patient_markers):
        next_pm_top = patient_markers[i + 1]["top"] if i + 1 < len(patient_markers) else None
        match = next(
            (t for t in total_markers
             if t["top"] > pm["top"] and (next_pm_top is None or t["top"] < next_pm_top)),
            None,
        )
        if match:
            segments.append({**pm, "end": match["bottom"] + BOTTOM_PADDING_PT, "complete": True})
        else:
            trimmed_end = _last_row_bottom(page, pm["top"], footer_top)
            segments.append({**pm, "end": trimmed_end, "complete": False})
    return segments


def crop_claim_tables(pdf_path, out_dir, dpi=DPI):
    os.makedirs(out_dir, exist_ok=True)
    saved = []  # each entry: {"path":..., "regions": [(page, top, bottom), ...]}

    with pdfplumber.open(pdf_path) as pdf:
        scale = dpi / 72.0
        pending = None

        for pno, page in enumerate(pdf.pages, start=1):
            segments = _page_segments(page)
            if not segments:
                pending = None
                continue

            page_img = page.to_image(resolution=dpi).original.convert("RGB")
            width_px = page_img.width
            seg_iter = iter(segments)

            if pending is not None:
                first = next(seg_iter)
                if first["claim"] != pending["claim"]:
                    saved.append({"path": _save(pending["crop"], pending, out_dir),
                                  "regions": pending["regions"]})
                    pending = None
                    top_px, end_px = int(first["top"] * scale), int(first["end"] * scale)
                    crop = page_img.crop((0, top_px, width_px, end_px))
                    if first["complete"]:
                        saved.append({"path": _save(crop, first, out_dir),
                                      "regions": [(page, first["top"], first["end"])]})
                    else:
                        pending = {**first, "crop": crop, "regions": [(page, first["top"], first["end"])]}
                else:
                    # Same claim continuing on this page: skip past the repeated
                    # "Claim Detail" title / patient-provider header / column
                    # header row instead of stitching from the very top of the
                    # page, so the header block isn't duplicated in the merged
                    # image.
                    header_bottom = _header_row_bottom(page, first["top"], first["end"])
                    start_px = int(header_bottom * scale)
                    end_px = int(first["end"] * scale)
                    part2 = page_img.crop((0, start_px, width_px, end_px))
                    merged = _stack(pending["crop"], part2)
                    regions = pending["regions"] + [(page, header_bottom, first["end"])]
                    if first["complete"]:
                        saved.append({"path": _save(merged, first, out_dir), "regions": regions})
                        pending = None
                    else:
                        pending = {**first, "crop": merged, "regions": regions}
                    if not first["complete"]:
                        continue

            for seg in seg_iter:
                top_px, end_px = int(seg["top"] * scale), int(seg["end"] * scale)
                crop = page_img.crop((0, top_px, width_px, end_px))
                if seg["complete"]:
                    saved.append({"path": _save(crop, seg, out_dir),
                                  "regions": [(page, seg["top"], seg["end"])]})
                else:
                    pending = {**seg, "crop": crop, "regions": [(page, seg["top"], seg["end"])]}

        if pending is not None:
            saved.append({"path": _save(pending["crop"], pending, out_dir), "regions": pending["regions"]})

    return saved


def _stack(img_top, img_bottom):
    w = max(img_top.width, img_bottom.width)
    h = img_top.height + img_bottom.height
    canvas = Image.new("RGB", (w, h), "white")
    canvas.paste(img_top, (0, 0))
    canvas.paste(img_bottom, (0, img_top.height))
    return canvas


def _save(img, seg, out_dir):
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", seg["name"]).strip("_")
    fname = f"{safe_name}_{seg['claim'] or 'noclaim'}.png"
    path = os.path.join(out_dir, fname)
    img.save(path)
    return path


# ----------------------------------------------------------------------
# 2. Rotate every page except the first two
# ----------------------------------------------------------------------
def rotate_pages(pdf_path, out_path, angle=ROTATE_ANGLE, skip_pages=SKIP_PAGES):
    reader = PdfReader(pdf_path)
    writer = PdfWriter()
    for i, page in enumerate(reader.pages):
        if i >= skip_pages:
            page.rotate(angle)
        writer.add_page(page)
    with open(out_path, "wb") as f:
        writer.write(f)
    return out_path


PROMPT = r"""
ROLE:
You are a highly accurate OCR and table extraction model specialized in
US dental Explanation of Benefits (EOB) documents.

TASK:
Extract the patient information, provider information, every service row,
and the Totals row from the dental EOB table shown in the image.

Return ONLY valid JSON.
Do not return markdown.
Do not return explanations.
Do not return ```json.
Do not return any text outside the JSON object.


=========================================================
PATIENT / PROVIDER INFORMATION
=========================================================

Extract:

1. Patient Name:
   Extract the value printed immediately after "Patient Name:".

2. Provider Name:
   Extract the value printed immediately after "Provider Name:".

Do NOT extract:
- Member #
- Member Type
- DOB
- Provider NPI
- Location NPI
- Place of Service
- Service Address
- Office Reference #
- Group
- Sub-Group
- Product
- Claim #
- Auth #
- Referral #
- Referral Date


=========================================================
SERVICE TABLE
=========================================================

Extract EVERY physical service row in the table.

The table columns are:

Item
Submitted Code
Paid Code
Tooth
Description
Date of Service
Submitted
Approved
Allowed
Other Insurance
Copay
Plan %
Deductible
Patient Pay
Writeoff
Plan Pay
Processing Policies


=========================================================
FIELDS TO EXTRACT FOR EACH SERVICE
=========================================================

For every physical service row extract ONLY these fields:

- item
- submitted_code
- paid_code
- date_of_service
- submitted
- approved
- allowed
- other_insurance
- copay
- deductible
- patient_pay
- writeoff
- plan_pay


DO NOT extract:

- tooth
- description
- plan %
- processing policies


=========================================================
COLUMN MAPPING
=========================================================

Item
-> item

Submitted Code
-> submitted_code

Paid Code
-> paid_code

Date of Service
-> date_of_service

Submitted
-> submitted

Approved
-> approved

Allowed
-> allowed

Other Insurance
-> other_insurance

Copay
-> copay

Deductible
-> deductible

Patient Pay
-> patient_pay

Writeoff
-> writeoff

Plan Pay
-> plan_pay


=========================================================
STRICT EXTRACTION RULES
=========================================================

1. Extract EVERY physical service row.
2. NEVER skip a service row.
3. NEVER duplicate a service row.
4. Preserve the original row order.
5. The Item value must come from the Item column.
6. Submitted Code must come ONLY from the Submitted Code column.
7. Paid Code must come ONLY from the Paid Code column.
8. Date of Service must come ONLY from the Date of Service column.
9. Submitted must come ONLY from the Submitted column.
10. Approved must come ONLY from the Approved column.
11. Allowed must come ONLY from the Allowed column.
12. Other Insurance must come ONLY from the Other Insurance column.
13. Copay must come ONLY from the Copay column.
14. Deductible must come ONLY from the Deductible column.
15. Patient Pay must come ONLY from the Patient Pay column.
16. Writeoff must come ONLY from the Writeoff column.
17. Plan Pay must come ONLY from the Plan Pay column.
18. NEVER shift a value from one column to another.
19. NEVER use values from another row.
20. NEVER use values from the Totals row as service-row values.
21. NEVER calculate a missing value.
22. NEVER infer a missing value.
23. NEVER correct a value mathematically.
24. If a cell is blank or unreadable, return "".
25. Preserve monetary values exactly as printed.
26. Preserve the "$" sign when it is visible.
27. Preserve commas and decimal places.
28. Do not convert "$62.00" to "62".
29. Do not convert "04/28/26" to another date format.
30. Do not extract the Description text.
31. Do not extract Tooth values.
32. Do not extract Plan %.
33. Do not extract Processing Policies.


=========================================================
IMPORTANT ROW RULE
=========================================================

A service row is a physical row in the table.

For example:

1 D0120 D0120 ...
2 D0330 D0330 ...
3 D1310 D1310 ...
4 D1330 D1330 ...

means there are FOUR service rows.

Even if two rows have identical values, they must remain separate rows.

Never merge two physical rows.


=========================================================
TOTALS
=========================================================

The row labeled "Total:" is NOT a service row.

Extract it separately into "totals".

Extract ONLY:

- submitted
- approved
- allowed
- other_insurance
- copay
- deductible
- patient_pay
- writeoff
- plan_pay


Totals mapping:

Total Submitted
-> submitted

Total Approved
-> approved

Total Allowed
-> allowed

Total Other Insurance
-> other_insurance

Total Copay
-> copay

Total Deductible
-> deductible

Total Patient Pay
-> patient_pay

Total Writeoff
-> writeoff

Total Plan Pay
-> plan_pay


Do NOT include:
- item
- submitted_code
- paid_code
- tooth
- description
- date_of_service
- plan %
- processing policies


=========================================================
CONFIDENCE
=========================================================

Every extracted field must contain:

{
    "value": "",
    "confidence": 0.0
}

Confidence must be between 0.0 and 1.0.

Confidence represents ONLY the visual certainty that the value
is actually present in the specified location.

1.0
= clearly visible and certain.

0.90 - 0.99
= clearly readable with very minor uncertainty.

0.80 - 0.89
= readable but slight visual uncertainty.

0.50 - 0.79
= partially unclear or difficult to read.

0.10 - 0.49
= very unclear.

0.0
= blank, missing, or unreadable.

If value is "":
confidence MUST be 0.0.

Do NOT increase confidence because a value is mathematically expected.

Do NOT use mathematical consistency to determine field confidence.

For example, if Submitted is "$62.00", return the visual confidence
that "$62.00" is actually visible in the Submitted column.


=========================================================
OUTPUT FORMAT
=========================================================

Return EXACTLY this structure:

{
  "patient_name": {
    "value": "",
    "confidence": 0.0
  },

  "provider_name": {
    "value": "",
    "confidence": 0.0
  },

  "services": [
    {
      "item": { "value": "", "confidence": 0.0 },
      "submitted_code": { "value": "", "confidence": 0.0 },
      "paid_code": { "value": "", "confidence": 0.0 },
      "date_of_service": { "value": "", "confidence": 0.0 },
      "submitted": { "value": "", "confidence": 0.0 },
      "approved": { "value": "", "confidence": 0.0 },
      "allowed": { "value": "", "confidence": 0.0 },
      "other_insurance": { "value": "", "confidence": 0.0 },
      "copay": { "value": "", "confidence": 0.0 },
      "deductible": { "value": "", "confidence": 0.0 },
      "patient_pay": { "value": "", "confidence": 0.0 },
      "writeoff": { "value": "", "confidence": 0.0 },
      "plan_pay": { "value": "", "confidence": 0.0 }
    }
  ],

  "totals": {
    "submitted": { "value": "", "confidence": 0.0 },
    "approved": { "value": "", "confidence": 0.0 },
    "allowed": { "value": "", "confidence": 0.0 },
    "other_insurance": { "value": "", "confidence": 0.0 },
    "copay": { "value": "", "confidence": 0.0 },
    "deductible": { "value": "", "confidence": 0.0 },
    "patient_pay": { "value": "", "confidence": 0.0 },
    "writeoff": { "value": "", "confidence": 0.0 },
    "plan_pay": { "value": "", "confidence": 0.0 }
  }
}


=========================================================
FINAL REQUIREMENT
=========================================================

Return ONLY valid JSON.

For every extracted field, return:
   - value
   - confidence

VALUE + CONFIDENCE RULES:

For every field return:
{
  "value": "",
  "confidence": ""
}

VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""

CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.

IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.

If value = "":
confidence MUST = 0.0.

Return ONLY the JSON.
"""


def extract_table_from_image(image_path, expected_rows=None):
    image = Image.open(image_path).convert("RGB")

    row_instruction = ""
    if expected_rows is not None:
        row_instruction = f"""

=========================================================
PHYSICAL ROW COUNT
=========================================================

Independent PDF analysis detected EXACTLY {expected_rows}
physical service rows in this image.

Therefore:

- Return EXACTLY {expected_rows} objects inside "services".
- Do NOT return fewer rows.
- Do NOT return more rows.
- Do NOT count the Total row.
- Do NOT count the header as a service row.
- Do NOT merge rows.
- Do NOT remove duplicate physical rows.

The number of objects in "services" MUST be exactly {expected_rows}.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT + row_instruction}
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=15000,
        temperature=0.0,
        repetition_penalty=1.1
    )

    generated_text = output[0]["generated_text"]
    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    generated_text = generated_text.strip()
    generated_text = re.sub(r"^```(?:json)?\s*", "", generated_text, flags=re.IGNORECASE)
    generated_text = re.sub(r"\s*```$", "", generated_text)

    return generated_text.strip()


def parse_amount(val):
    if val in ["", None]:
        return 0.0
    return float(str(val).replace("$", "").replace(",", "").strip())


SERVICE_FIELDS = [
    "item", "submitted_code", "paid_code", "date_of_service", "submitted",
    "approved", "allowed", "other_insurance", "copay", "deductible",
    "patient_pay", "writeoff", "plan_pay"
]

TOTAL_FIELDS = [
    "submitted", "approved", "allowed", "other_insurance", "copay",
    "deductible", "patient_pay", "writeoff", "plan_pay"
]


def clean_confidence_field(value):
    if isinstance(value, dict):
        try:
            confidence = float(value.get("confidence", 0.0))
        except (ValueError, TypeError):
            confidence = 0.0
        confidence = max(0.0, min(1.0, confidence))
        return {"value": str(value.get("value", "")), "confidence": confidence}

    if value is None:
        return {"value": "", "confidence": 0.0}

    return {"value": str(value), "confidence": 0.0}


def enforce_schema(parsed):
    cleaned = {
        "patient_name": clean_confidence_field(parsed.get("patient_name", "")),
        "provider_name": clean_confidence_field(parsed.get("provider_name", "")),
        "services": [],
        "totals": {},
    }

    for row in parsed.get("services", []):
        new_row = {field: clean_confidence_field(row.get(field, "")) for field in SERVICE_FIELDS}
        cleaned["services"].append(new_row)

    totals = parsed.get("totals", {})
    for field in TOTAL_FIELDS:
        cleaned["totals"][field] = clean_confidence_field(totals.get(field, ""))

    return cleaned


def validate_eob_table(parsed, patient_name, expected_row_count):
    services = parsed.get("services", [])
    totals = parsed.get("totals", {})
    fields = ["submitted", "approved", "allowed", "other_insurance",
              "copay", "deductible", "patient_pay", "writeoff", "plan_pay"]

    if not services:
        errors = [{"field": f, "computed": 0.0, "extracted": None} for f in fields]
        return False, "No services found", [{"error": "empty services"}] + errors, len(fields)

    computed_totals = {
        f: round(sum(parse_amount(get_field_value(r.get(f, ""))) for r in services), 2)
        for f in fields
    }

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    for field, computed_value in computed_totals.items():
        extracted_value = round(parse_amount(get_field_value(totals.get(field, ""))), 2)
        diff = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01

        if match:
            icon, status = "✅", "MATCH"
        else:
            icon, status = "❌", "MISMATCH"
            has_error = True
            errors.append({
                "type": "field_mismatch",
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value,
                "difference": diff
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    extracted_row_count = len(services)
    if expected_row_count == extracted_row_count:
        icon, status = "✅", "MATCH"
    else:
        icon, status = "❌", "MISMATCH"
        has_error = True
        errors.append({
            "type": "row_count_mismatch",
            "expected_rows": expected_row_count,
            "extracted_rows": extracted_row_count
        })

    line = f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} | extracted={extracted_row_count:<10} {status}"
    print(line)
    result_validation += "\n" + line
    print("-" * 80)

    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors, len(fields)
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, [], len(fields)


def count_service_rows(regions):
    """
    regions: list of (page, region_top, region_bottom) tuples.
    For a single-page block, pass one region.
    For a merged (cross-page) block, pass one region per page:
        [(prev_page, start_y, prev_page.height), (curr_page, header_bottom, end_y)]
    """
    total = 0

    for page, region_top, region_bottom in regions:
        words = page.extract_words()
        row_positions = []

        for w in words:
            text = w["text"].strip()
            match = re.search(r"\bD\d{4}\b", text)
            if match:
                y = float(w["top"])
                if region_top <= y <= region_bottom:
                    row_positions.append(y)

        row_positions.sort()

        grouped_rows = []
        threshold = 3
        for y in row_positions:
            if not grouped_rows or abs(y - grouped_rows[-1]) > threshold:
                grouped_rows.append(y)

        total += len(grouped_rows)

    return total


def normalize_repeated_chars(text):
    if not text:
        return ""
    return re.sub(r"(.)\1+", r"\1", text)


def get_field_value(field):
    if isinstance(field, dict):
        return field.get("value", "")
    return field or ""


def check_claim_denied(pdf_path):
    denial_keywords = ["denied", "denial"]
    skip_phrases = ["important information about your"]

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            text = normalize_repeated_chars(text).lower()

            if any(phrase in text for phrase in skip_phrases):
                continue

            for keyword in denial_keywords:
                if keyword in text:
                    print(f"❌ Claim denied: '{keyword}' found on page {page_num}")
                    return "denied"

    print("✅ No denied keyword found")
    return "not denied"


def run_pipeline(pdf_path, output_dir="EOB_OUTPUT/DentaQuest", company_name="DentaQuest"):
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    crop_dir = os.path.join("claim_crops", pdf_name)
    os.makedirs(crop_dir, exist_ok=True)

    print("=" * 80)
    print(f"Processing: {os.path.basename(pdf_path)}")
    print("=" * 80)

    claim_status = check_claim_denied(pdf_path)
    print(f"Claim status: {claim_status}")

    crops = crop_claim_tables(pdf_path, crop_dir)
    print(f"Created {len(crops)} claim crops")

    all_patients = []

    for entry in crops:
        image_path, regions = entry["path"], entry["regions"]
        print(f"\nProcessing crop: {image_path}")

        try:
            llm_output = extract_table_from_image(image_path)
            parsed = json.loads(llm_output)

            model_confidence = calculate_model_confidence(parsed)
            parsed = enforce_schema(parsed)

            expected_rows = count_service_rows(regions)
            extracted_rows = len(parsed.get("services", []))
            patient_name = get_field_value(parsed.get("patient_name", ""))

            validation_status, val_log, total_errors, total_fields = validate_eob_table(
                parsed, patient_name=patient_name, expected_row_count=expected_rows
            )

            patient = {
                "patient_name": patient_name,
                "provider_name": get_field_value(parsed.get("provider_name", "")),
                "services": parsed.get("services", []),
                "totals": parsed.get("totals", {}),
                "validation": {
                    "status": validation_status,
                    "errors": total_errors
                },
                "_model_confidence": model_confidence
            }

            all_patients.append(patient)

            print(f"Patient: {patient['patient_name']}")
            print(f"Provider: {patient['provider_name']}")
            print(f"Rows: {extracted_rows}/{expected_rows}")
            print(f"Validation: {'PASS' if validation_status else 'FAIL'}")
            print(f"Model confidence: {model_confidence * 100:.2f}%")

        except Exception as e:
            print(f"❌ Error processing {image_path}: {e}")

            all_patients.append({
                "patient_name": "",
                "provider_name": "",
                "services": [],
                "totals": {},
                "validation": {
                    "status": False,
                    "errors": [{"type": "extraction_error", "message": str(e)}]
                },
                "_model_confidence": 0.0
            })

    confidence_score = calculate_eob_confidence(all_patients)

    for patient in all_patients:
        patient.pop("_model_confidence", None)

    final = [{
        "eob_id": pdf_name,
        "file_name": os.path.basename(pdf_path),
        "claim_status": claim_status,
        "payor": "DentaQuest",
        "confidence_score": confidence_score,
        "patients": all_patients
    }]

    success_path, failed_path = save_split_output(
        final,
        company_name="DentaQuest",
        pdf_name=pdf_name,
        pdf_path=pdf_path,
        cropped_dir=crop_dir
    )

    print("\n" + "=" * 80)
    print(f"Final confidence: {confidence_score:.2f}%")
    print(f"Success: {success_path}")
    print(f"Failed: {failed_path}")
    print("=" * 80)

    return final

W0909 13:09:09.688000 350371 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0909 13:09:09.707000 350371 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


ModuleNotFoundError: No module named 'output_utils'